# Chat Completion API로 프롬프트 엔지니어링 실습

Chat Completion API는 system, user, assistant 역할을 가진 메시지 목록을 모델에 전달하고 생성 응답을 받는 인터페이스이다. system 메시지는 모델의 역할과 공통 규칙을, user 메시지는 현재 요청과 입력 데이터를, assistant 메시지는 대화 이력을 표현한다. API 호출은 프롬프트를 코드로 재사용하고 결과 형식을 자동 처리할 수 있게 해 준다.

프롬프트 엔지니어링은 단순히 질문을 길게 쓰는 작업이 아니다. 역할, 입력 경계, 처리 규칙, 출력 스키마를 분리해 모델이 따라야 할 계약을 명확히 하는 작업이다. 강점은 빠른 실험과 다양한 업무 적용이며, 한계는 모델 출력이 확률적이고 외부 API 비용·속도·보안 제약이 있다는 점이다. API 키는 코드나 노트북에 저장하지 않고 실행 환경의 비밀 저장소에서 읽어야 한다.

이번 실습은 API 클라이언트를 준비한 뒤 기사 제목 교정, 상담형 응답, 레시피 제안, JSON 면접 질문 생성에 같은 메시지 구조가 어떻게 재사용되는지 확인한다. 외부 API를 호출하는 셀은 키·네트워크·모델 권한이 준비된 환경에서만 실행하고, 응답 내용은 업무 규칙과 JSON 파싱 결과로 다시 검증한다.


### OpenAI Python SDK 설치

이 셀은 Chat Completion API를 호출하기 위한 `openai` 패키지를 설치한다. 설치가 끝나면 이후 셀의 `OpenAI` 클래스를 가져올 수 있다. 패키지 버전과 인터넷 연결이 필요한 환경 준비 단계이므로, 이미 설치된 수업 환경에서는 실행 결과가 달라질 수 있다.


In [1]:
%pip install -U openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## PyCharm 환경 설정

1. PyCharm에서 `08_llm` 폴더를 프로젝트로 연다.
2. PyCharm Terminal에서 `python -m pip install openai python-dotenv`를 실행한다.
3. `08_llm/.env` 파일에 `OPENAI_API_KEY`를 저장한다.
4. 키 값은 코드셀, 출력, Git 또는 공유 파일에 포함하지 않는다.


### PyCharm 프로젝트의 `.env` 설정 불러오기

`.env`는 API 키와 실행 설정을 노트북 코드에서 분리하는 로컬 파일이다. `find_dotenv(usecwd=True)`는 현재 Jupyter 작업 폴더부터 상위 폴더로 이동하며 `08_llm/.env`를 찾고, `load_dotenv()`는 그 값을 현재 커널의 환경 변수로 불러온다.

이 셀은 `OPENAI_API_KEY` 변수가 준비되었는지만 검사하고 실제 값은 출력하지 않는다. 이후 OpenAI SDK와 연동 라이브러리는 환경 변수를 자동으로 사용한다.


In [2]:

import os

from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError(
        "08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 현재 셀을 다시 실행하세요."
    )

load_dotenv(dotenv_path, override=False)

required_env_vars = [
    "OPENAI_API_KEY",
]

missing_env_vars = [name for name in required_env_vars if not os.getenv(name)]
if missing_env_vars:
    
    missing_names = ", ".join(missing_env_vars)
    raise RuntimeError(f".env 파일의 다음 변수를 확인하세요: {missing_names}")

print("환경 변수 준비 완료")

환경 변수 준비 완료


### API 클라이언트 초기화

`OpenAI` 객체는 이후 모든 API 요청의 진입점이다. 앞 셀에서 읽은 키를 `api_key`에 전달해 `client`에 저장하고, 뒤의 프롬프트 함수들이 이 클라이언트를 재사용한다. 객체 생성은 네트워크 요청 자체가 아니지만, 실제 호출은 유효한 키와 사용 권한을 요구한다.


In [3]:
from openai import OpenAI

client = OpenAI()

### 가장 작은 Chat Completion 요청 만들기

이 셀은 `messages` 목록 안에 system과 user 메시지를 넣어 모델에 한 번 요청한다. `model`은 사용할 모델 이름, `temperature`는 생성 다양성, `max_completion_tokens`는 생성 길이 상한을 뜻한다. 응답 객체는 다음 셀에서 `choices[0].message.content`로 꺼낸다. 외부 API 호출이므로 실행하면 인증·네트워크·모델 사용 가능 여부를 함께 확인한다.


In [4]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",

    # 전달할 대화
    messages=[
        {
            "role":"system", # 모델이 대화 전체에서 따를 역할과 규칙 작성
            "content": [
                {
                    "type": "text",
                    "text": "너는 아주 친절하고, 많은 도움을 주는 챗봇이야"
                }
            ]
        },
        {
            "role": "user", # 사용자가 전달할 실제 입력 값
            "content": [
                {
                    "type": "text",
                    "text": "안녕~ 내 이름은 아이유야~"
                }
            ]
        }
    ],

    # 응답 형식 지정 "text", "json"
    response_format={"type": "text"},
    temperature=1, # 온도 값이 높을수록 후보 토큰 수가 많아져 무작위성 증가
    max_completion_tokens=2048, # 텍스트 생성 상한
    top_p=1, # 누적 확률 범위 제한 X
    frequency_penalty=0, # 같은 표현 반복 불이익 점수
    presence_penalty=0, # 이미 등장한 주제를 다시 선택하는 경우 불이익 점수
)

### 응답 객체에서 생성 텍스트 추출하기

응답의 `choices`는 생성 후보 목록이며, 이 예제는 첫 번째 후보의 `message.content`를 표시한다. 후보가 비어 있거나 API 호출이 실패하면 이 접근은 불가능하므로, 앞 셀의 정상 응답 여부를 먼저 확인해야 한다. 출력 텍스트는 모델의 생성물이지 검증된 사실이 아니므로 업무 규칙과 원문을 대조한다.


In [5]:
print(response.choices[0].message.content)

안녕, 아이유! 만나서 반가워요. 어떻게 도와줄까요? 😊


## 프롬프팅의 기본구성

https://www.deeplearning.ai/short-courses/chatgpt-prompt-engineering-for-developers/

1. Instruction 지시사항
2. Context 문맥
3. Input Data/Example 입력/예시
4. Output Indicator 출력지시

## 기사 제목 교정

- 기자들이 송고한 기사에서 제목을 추출하고, 표현 조정
- 프랑스AFP 속보시스템에서 도입되어 사용


### 기사 제목 교정 프롬프트 실행

이 셀은 역할과 교정 규칙을 `system_message`에, 실제 기사 제목을 `user_message`에 분리해 전달한다. 예시와 출력 형식은 모델이 두 제목을 정해진 구조로 반환하도록 돕는다. `print` 결과에서 비속어 완화, 핵심 정보 보존, 지정된 두 줄 형식이 모두 지켜졌는지 확인한다. API 응답은 실행 환경에 따라 달라지며 결과를 실제 관찰한 값처럼 미리 단정하지 않는다.


In [7]:
# 교정이 필요한 제목
# title_before = '테이의 FM 개꿀 라디오 방송에 주목해주세요.'
title_before = '졸라 빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환'

# system 지시사항
# system 메시지에는 모든 제목에 공통으로 적용할 역할, 절차, 출력 형식, 예시를 넣는다.
system_message = """
기사들이 송고한 제목에서 맞춤법, 문법, 의미, 어조등에 있어서 교정작업을 수행해 주세요.

- 기사 제목이 명확하고 주제와 잘 맞도록 조정하세요.
- 독자의 관심을 끌 수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.
- 어조가 지나치게 감정적이거나 부정적인 경우 표현을 완화하거나, 중립적인 어조로 수정하세요.
- 비속어가 포함되어 있는 경우, 비속어를 반드시 제거하고, 의미를 적절히 유지하도록 제목을 교정하세요.

### Steps ###
1. 기사제목을 읽고 주요내용을 이해하세요.
2. 제목이 전달하고자하는 메세지를 명확하게 반영하는지 검토하세요.
3. 맞춤법, 문법, 의미 전달의 정확성등을 점검하고 적절히 수정하세요.
4. 제목이 자연스럽고, 독자에게 매력적으로 다가갈수 있는지 점검하고 매우 간결하게 정리하세요.


### Output Format ###
기사 원래 제목과 교정된 제목을 다음 형식으로 제공하세요.

- 원래 제목: [기사 원래 제목]
- 교정 제목: [교정 기사 제목]

### Examples ###
- 원래 제목: "어제 서울에서 큰 불이 나 수백명이 대피했다."
- 교정 제목: "서울 대형 화재, 수백명 대피"

- 원래 제목: "전기자동차 판매량 급감에 내연차회사들이 즐거워하는 중입니다."
- 교정 제목: "전기차 판매량 급감에 웃는 내연차회사들"

### Extra Instructions ###
- 제목이 너무 길면, 간결하게 줄이되, 핵심 메세지를 잃어버려서는 안됩니다.
- 지역명, 시간 등의 중요한 정보는 명확하게 유지하세요.
- 제목이 특정집단이나 대상에 대해 중립적이지 않을 경우, 그 표현을 완화하세요.

"""

# 이번 요청에서만 바뀌는 기사 제목을 user 메시지에 삽입
user_message = f"""
다음 기사제목을 교정해주세요.

제목: {title_before}
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",  # 제목 교정에 사용할 모델 ID이다.
    messages=[
        {
            "role": "system",  # 위에서 만든 공통 교정 규칙을 전달한다.
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ]
        },
        {
            "role": "user",  # 교정할 제목이 포함된 현재 요청을 전달한다.
            "content": [
                {
                    "type": "text",
                    "text": user_message
                }
            ]
        }
    ],
    response_format={
        "type": "text"  # 결과를 일반 문자열로 받는다.
    },
    temperature=1,  # 표현의 다양성을 조절한다.
    max_completion_tokens=2048,  # 교정 결과가 사용할 수 있는 생성 토큰 상한이다.
    top_p=1,  # 후보 토큰의 누적 확률 범위를 제한하지 않는다.
    frequency_penalty=0,  # 동일 표현 반복에 대한 추가 패널티를 사용하지 않는다.
    presence_penalty=0  # 새로운 주제 사용을 강제로 유도하지 않는다.
)

# 응답 결과 확인
print(response.choices[0].message.content)

- 원래 제목: 졸라 빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환
- 교정 제목: 힘든 작업으로 끼니 거르기 잦은 노동자들의 어려움


### 반복 요청을 제목 교정 함수로 리팩터링하기

`correct_news_title`은 제목, 모델 이름, `temperature`, `top_p`를 인자로 받아 같은 프롬프트 구조를 재사용한다. 함수의 반환값은 응답 텍스트이며, 호출한 쪽은 `output`에 저장해 출력하거나 후속 검증에 사용한다. 같은 제목을 여러 설정으로 비교할 때는 입력을 고정하고 생성 파라미터만 바꿔 형식 준수와 표현 차이를 비교한다.


In [8]:
def correct_news_title(title_before, model='gpt-4o-mini', temperature=1, top_p=1):
    system_message = """
    기사들이 송고한 제목에서 맞춤법, 문법, 의미, 어조등에 있어서 교정작업을 수행해 주세요.

    - 기사 제목이 명확하고 주제와 잘 맞도록 조정하세요.
    - 독자의 관심을 끌 수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.
    - 어조가 지나치게 감정적이거나 부정적인 경우 표현을 완화하거나, 중립적인 어조로 수정하세요.
    - 비속어가 포함되어 있는 경우, 비속어를 반드시 제거하고, 의미를 적절히 유지하도록 제목을 교정하세요.

    ### Steps ###
    1. 기사제목을 읽고 주요내용을 이해하세요.
    2. 제목이 전달하고자하는 메세지를 명확하게 반영하는지 검토하세요.
    3. 맞춤법, 문법, 의미 전달의 정확성등을 점검하고 적절히 수정하세요.
    4. 제목이 자연스럽고, 독자에게 매력적으로 다가갈수 있는지 점검하고 매우 간결하게 정리하세요.


    ### Output Format ###
    기사 원래 제목과 교정된 제목을 다음 형식으로 제공하세요.

    - 원래 제목: [기사 원래 제목]
    - 교정 제목: [교정 기사 제목]

    ### Examples ###
    - 원래 제목: "어제 서울에서 큰 불이 나 수백명이 대피했다."
    - 교정 제목: "서울 대형 화재, 수백명 대피"

    - 원래 제목: "전기자동차 판매량 급감에 내연차회사들이 즐거워하는 중입니다."
    - 교정 제목: "전기차 판매량 급감에 웃는 내연차회사들"

    ### Extra Instructions ###
    - 제목이 너무 길면, 간결하게 줄이되, 핵심 메세지를 잃어버려서는 안됩니다.
    - 지역명, 시간 등의 중요한 정보는 명확하게 유지하세요.
    - 제목이 특정집단이나 대상에 대해 중립적이지 않을 경우, 그 표현을 완화하세요.

    """

    user_message = f"""
    다음 기사제목을 교정해주세요.

    제목: {title_before}
    """

    response = client.chat.completions.create(
        model=model,  # 호출자가 선택한 모델 ID이다.
        messages=[
            {
                "role": "system",
                "content": [
                    {
                    "type": "text",
                    "text": system_message
                    }
                ]
            },
            {
                "role": "user",  # 이번에 교정할 제목이다.
                "content": [
                    {
                    "type": "text",
                    "text": user_message
                    }
                ]
            }],
            response_format={
                "type": "text"
            },
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )
    # 호출부가 SDK 응답 구조를 몰라도 되도록 생성된 문자열만 반환한다.
    return response.choices[0].message.content

title_before = '주말 미친 폭우 예상, 모두들 무사하시길~'
output = correct_news_title(title_before)
print(output)

- 원래 제목: 주말 미친 폭우 예상, 모두들 무사하시길~
- 교정 제목: "주말 폭우 예보, 안전 유의하세요"


## 연애코치 ReAct

### ReAct 형식의 상담 프롬프트 함수 정의

ReAct는 추론과 행동을 명시적으로 구분해 복잡한 작업을 단계적으로 수행하게 하는 프롬프팅 방식이다. 이 예제는 외부 도구 호출을 구현하지 않고 상황 분석·행동 계획·실행이라는 출력 구조를 요청한다. `dating_coach`는 사용자 고민 문자열을 받아 응답 텍스트를 돌려주며, 다음 두 셀이 서로 다른 입력에서 구조가 유지되는지 확인한다.


In [4]:
def dating_coach(prompt, model='gpt-4o-mini', temperature=1, top_p=1):
    # System message: 지침 설정
    system_message = """
    << System Instruction >>
    어떤 상황에서든 최고의 논리적/감성적 관점을 적용하는 연애코치로써 사용자의 고민을 해결해 주세요.

    << Output Format >>
    1. 상황분석:

    2. 행동계획:

    3. 실행:
    """

    # User message: 사용자 입력 프롬프트를 일정 형식으로 가공한 형태
    user_message = f"""
    사용자 현재 현황:
    {prompt}
    """

    # Chat Completion 요청
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role":"system",
                "content": [{"type": "text", "text": system_message}]
             },
            {
                "role":"user",
                "content": [{"type": "text", "text": user_message}]
            },
        ],
        response_format={"type": "text"}, # 응답 타입 지정
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    # 첫 번째 후보의 상담 결과 텍스트만 반환
    return response.choices[0].message.content

### 첫 번째 상담 입력으로 출력 형식 확인

정의한 `dating_coach`에 기념일 선물이라는 한 문장 입력을 전달한다. 출력에서 상황 분석, 행동 계획, 실행 항목이 구분되는지와 안전하지 않거나 과도하게 단정적인 조언이 없는지를 확인한다. 실제 결과는 모델과 실행 시점에 따라 달라진다.


In [5]:
prompt = """
저 이번 주말에 여자친구와 100일인데, 기억에 남는 선물을 하고 싶다. 뭐가 좋을까?
"""

print(dating_coach(prompt))

1. 상황분석:
   - 100일이라는 특별한 기념일을 맞이하고 있으며, 이 날을 기념하기 위해 여자친구에게 특별하고 기억에 남는 선물을 하기를 원하고 있습니다.
   - 여자친구의 취향이나 관심사를 반영한 선물이 좋은 선택이 될 것입니다. 그렇기 때문에 그녀의 선호도와 함께 의미 있는 선물로 그녀에게 감동을 줄 수 있는 방법을 고민해야 합니다.

2. 행동계획:
   - 여자친구의 취향, 관심사, 그리고 두 사람의 특별한 기억이 담긴 요소를 고려하여 선물을 구상합니다. 
   - 아래와 같은 유형의 선물을 고려해볼 수 있습니다:
     - **의미 있는 아이템**: 처음 만났던 곳의 사진을 담은 액자, 두 분의 이야기가 담긴 작은 스크랩북 등.
     - **경험 선물**: 함께가는 드라이브, 특별한 레스토랑에서의 저녁 식사, 혹은 그녀가 좋아하는 공연의 티켓.
     - **손편지와 함께하는 선물**: 선물과 함께 진솔한 마음을 담아 손편지를 쓰면 더욱 감동적일 것입니다.

3. 실행:
   - 우선 그녀의 취향을 파악하여 위의 옵션 중 어떤 것이 가장 마음에 들지 고민해보세요.
   - 마음에 드는 선물이 정해지면 미리 구매하고, 또는 예약을 진행하시길 바랍니다.
   - 선물 준비가 끝난 후, 그녀를 위한 손편지를 작성해 감정과 감사의 마음을 전하세요.
   - 기념일 당일, 정성스럽게 준비한 선물을 주면서, 그동안의 소중하고 특별했던 순간들을 함께 회상하며 축하의 말도 잊지 마세요. 

이렇게 준비한다면 여자친구에게 깊은 감동을 줄 수 있을 것입니다.


### 두 번째 상담 입력으로 일반화 범위 확인

같은 함수를 다른 갈등 상황에 적용해 프롬프트 구조가 입력 변화에도 유지되는지 확인한다. 두 응답을 비교할 때는 문장 길이보다 상황에 맞는 행동 제안과 출력 형식 준수 여부를 기준으로 삼는다. 개인 관계 조언은 사실 판단이나 전문 상담을 대체하지 않는다는 한계도 함께 안내한다.


In [6]:
prompt = """
내가 친구들과 놀다가 여자친구 연락을 받지 못했어. 지금 단단히 삐져있는 상황이야
"""

print(dating_coach(prompt))

1. 상황분석:
   당신은 친구들과의 시간을 보내고 있을 때 여자친구의 연락을 받지 못해서 불안함과 불만을 느끼고 있습니다. 이러한 상황은 보통 상대방이 나를 신경 쓰지 않거나 상관하지 않는 것처럼 느껴져 소외당하는 기분을 유발할 수 있습니다. 감정적으로는 불안감과 서운함이 겹쳐 긴장이 커진 상태일 가능성이 큽니다.

2. 행동계획:
   - 먼저 마음을 가라앉히세요. 왜 그녀가 연락을 하지 않았는지, 혹시 바쁜 것은 아닌지 등을 고려해보세요.
   - 감정이 격해진 상태에서 즉흥적으로 행동하지 말고, 상대방에게 연락해보는 것을 계획하세요. 감정적으로 냉정하게 그녀와의 대화를 준비하는 것이 중요합니다.
   - 연락을 하는 방법은 부드럽게 관심을 보이는 형태로 진행하세요. 예를 들어, "오늘 바빴어? 아무 일 없는지 궁금해!" 같은 메시지를 보내보세요.

3. 실행:
   - 친구들과의 시간을 이어가면서도 마음 속의 불안감을 줄이는 데 집중하세요. 만약 친구들이 심리적으로 위로가 된다면 그들과의 대화에 집중하세요.
   - 이후 적절한 타이밍을 찾아 여자친구에게 메시지를 보내거나 전화를 걸어 부드럽고 긍정적인 톤으로 대화를 시작하세요. 자신의 감정도 솔직하게 표현하되, 비난이나 질책보다는 당신의 마음 상태를 이해해달라는 방식으로 접근하는 것이 효과적입니다.
   - 그녀의 반응에 따라 대화를 지속하며 서로의 입장을 이해해보려 노력하세요.


## 냉털마스터 ReAct
- 사용자는 냉장고에 남아있는 음식재료를 알려주면, LLM은 이를 바탕으로 어떤 음식을 만들지를 조언해준다.
- Reasoning/Action을 끌어낼수 있는 적절한 프롬프팅을 작성한다.


### 재료 기반 레시피 제안 함수 정의

이 함수는 재료 목록을 user 메시지에 넣고, 분석·계획·검증·최종 레시피 순서의 응답을 요청한다. 리스트인 `user_foods`는 f-string에서 문자열로 변환되어 프롬프트의 입력 맥락이 된다. 모델은 실제 식재료 상태나 알레르기를 알 수 없으므로, 생성한 레시피는 조리 전 안전성·보관 상태·알레르기 정보를 사용자가 다시 확인해야 한다.


In [7]:
def fridge_raid_master(user_foods, model='gpt-4o-mini', temperature=1, top_p=1):
    system_message = """
당신은 사용자 냉장고의 재료를 가지고 최고의 음식을 만들 수 있는 레시피를 추천하는 챗봇입니다.
현재 상황을 분석하고 실행계획을 세우며 추가 내용이 있을지 확인하는 꼼꼼함을 보여주세요.

## 지시사항

### 상황분석
1. **재료 확인**: 냉장고에 있는 재료 목록을 작성하고, 각 재료의 신선함과 사용 가능 시간을 파악합니다.
2. **요리 컨셉 설정**: 재료를 고려하여 요리의 주제를 정하고 잠재적인 요리 아이디어를 계획합니다.

### 실행계획
3. **목표 변경**: 사용 가능한 재료로 만들고자 하는 요리의 목표를 설정하고, 필요한 정보나 추가 자료를 조사합니다.
4. **전략 개발**: 단계별로 목표를 달성하기 위한 전반적인 요리 계획을 세웁니다.
    단, 사용자가 따라하기 쉽게 단계별 가이드를 작성해야 합니다.

### 검증 및 추가내용 확인
5. **레시피 검토**: 작성된 레시피를 검토하여 정보가 명확하고 완전한지 확인합니다.
6. **수정**: 필요 시 수정하여 최종 레시피를 완성합니다.

## 출력형식

1. 상황분석:
   - 재료 목록 및 상태
   - 요리 컨셉 및 아이디어

2. 실행계획:
   - 목표 및 세부 계획
   - 필요한 추가 자료 및 전략

3. 검증 및 추가내용 확인
   - 레시피의 정확성 검토
   - 수정 및 편집

4. 최종레시피
   - 사용자가 따라하기 쉽게 목록으로 작성
   - 필요한 준비물로 별도록 작성할것!

## Examples

- **Input**:
    사용자의 냉장고에는 현재 [무, 파, 두부]이/가 있습니다.
-   **Output**:
  1. 상황분석: 오늘은 냉장고에 있는 무, 파, 두부를 이용해 한식 스프를 만들어보겠습니다...
  2. 실행계획: 먼저 무와 파를 얇게 썰어 냄비에 넣고 물을 부어...
  3. 검증 및 추가내용 확인: 레시피를 확인한 결과, 무와 두부의 사용 방법에 대한 설명이 추가로 필요합니다...

## Notes

- 요리의 문화적 관련성과 감수성을 고려합니다.
- 레시피는 예상 독자의 요리 지식 수준에 맞게 조정됩니다.

    """

    user_message = f"""
    사용자의 냉장고에는 현재 {user_foods}이/가 있습니다.
    """
    # Chat Completion 요청
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role":"system",
                "content": [{"type": "text", "text": system_message}]
             },
            {
                "role":"user",
                "content": [{"type": "text", "text": user_message}]
            },
        ],
        response_format={"type": "text"}, # 응답 타입 지정
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    return response.choices[0].message.content

### 재료 목록을 Markdown 응답으로 표시하기

`Markdown`과 `display`는 모델이 반환한 마크다운 형식의 레시피를 노트북에서 읽기 쉽게 렌더링한다. `user_foods`가 함수 입력이고 반환 문자열이 `Markdown`의 입력이 된다. 화면에서는 네 개의 출력 구역이 실제로 구분되는지와 재료 목록이 모두 반영되었는지 확인한다.


In [9]:
from IPython.display import Markdown

user_foods = ['소고기 부챗살', '양파', '새송이 버섯', '명란젓', '배추김치', '된장']
display(Markdown(fridge_raid_master(user_foods)))

1. **상황분석**:
   - **재료 목록 및 상태**:
     - 소고기 부챗살: 신선하며 요리하기 적합한 상태
     - 양파: 신선하며 사용 가능
     - 새송이 버섯: 신선하며 사용 가능
     - 명란젓: 신선하며 사용 가능
     - 배추김치: 적당히 발효되어 활용 가능
     - 된장: 유통기한 내 사용 가능

   - **요리 컨셉 및 아이디어**:
     오늘은 냉장고에 있는 재료들을 이용해 '소고기와 새송이 버섯의 된장찌개'를 만들어보겠습니다. 된장찌개는 한국 전통 음식으로, 영양가도 높고 따뜻한 국물이 특징입니다. 명란젓과 배추김치를 곁들여 풍성한 한 끼를 완성할 수 있습니다.

2. **실행계획**:
   - **목표 및 세부 계획**:
     - 소고기 부챗살, 양파, 새송이 버섯을 주재료로 하여 된장찌개를 만드는 것이 목표입니다.
     - 1) 부챗살과 양파를 적당한 크기로 썰어 볶기
     - 2) 물과 된장을 넣어국물 만들기
     - 3) 새송이 버섯과 배추김치를 추가하여 끓이기
     - 4) 마지막으로 명란젓을 곁들이기

   - **필요한 추가 자료 및 전략**:
     - 기본 된장찌개 만드는 방법을 자세히 조사하고, 각 재료의 조리 시간을 동영상 자료나 블로그 포스트 등에서 확인하여 정확한 조리법을 확립해야 합니다.

3. **검증 및 추가내용 확인**:
   - **레시피의 정확성 검토**: 계획된 레시피에 따라 모든 재료를 올바르게 활용할 수 있으며, 각 단계의 설명이 분명하고 따라하기 쉬운지 확인했습니다.
   - **수정 및 편집**: 조리 시간과 불 조절에 대한 추가 정보를 넣기로 했습니다.

4. **최종레시피**:
   - **필요한 준비물**:
     - 소고기 부챗살 200g
     - 양파 1개
     - 새송이 버섯 2개
     - 된장 2큰술
     - 배추김치 적당량
     - 명란젓 50g
     - 물 4컵
     - 소금, 후추 (선택 사항)
  
   - **조리 방법**:
     1. 소고기 부챗살을 적당한 크기로 썰고 양파도 썰어 준비합니다.
     2. 냄비에 기름을 두르고 소고기와 양파를 넣어 볶습니다. (중불에서 5분)
     3. 물 4컵을 부어 끓입니다.
     4. 끓으면 된장을 풀어 넣고 잘 섞어 줍니다.
     5. 썰어 놓은 새송이 버섯을 추가하고 약 5분 더 끓입니다.
     6. 마지막으로 배추김치와 명란젓을 넣고 일정 시간 더 끓인다 (약 2~3분).
     7. 간을 보고 필요 시 소금과 후추로 간을 조절합니다.
     8. 뜨거운 상태로 그릇에 담아 서브합니다. 

이렇게 만들어진 소고기와 새송이 버섯의 된장찌개는 따뜻하고 깊은 맛이 있어, 명란젓과 배추김치와 함께 즐기면 더욱 맛있습니다!

## 면접질문 생성 JSON 출력


### JSON 형식 면접 질문 생성 함수 정의

이 함수는 채용공고 문자열을 받아 hard skill과 soft skill 질문·답변을 담은 JSON 문자열을 요청한다. `response_format={"type": "json_object"}`는 JSON 객체 형식의 응답을 요구하지만, 실제 파싱 전에는 필수 키와 값의 타입을 검증해야 한다. 함수는 아직 API를 호출하지 않으며 다음 셀의 채용공고가 입력으로 전달될 때 호출된다.


In [10]:
def job_interview(job_posting, model='gpt-4o-mini', temperature=1, top_p=1):
    system_message = """
당신은 머신러닝/딥러닝/LLM/AI서비스개발의 전문가로써, 해당분야의 으뜸가는 면접관입니다.
매번 그룹사의 인재를 발굴하기 위해 면접질문/모범답안을 작성하고 있습니다.

<<지시사항>>
- 사용자가 제출한 채용공고의 내용을 바탕으로 예상면접질문과 답변을 작성해주세요.
- 하드스킬과 소프트스킬 두개의 섹션으로 나누어 작성해주세요.
- 각 스킬별로 질문/답변을 3개씩 만들어주세요.

<<출력형식>>
출력은 json형식으로만 반환되어야 합니다.

{{
    "hard_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ],
    "soft_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ]
}}
"""

    user_message=f"""
    채용공고: {job_posting}
    """

    # Chat Completion 요청
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role":"system",
                "content": [{"type": "text", "text": system_message}]
             },
            {
                "role":"user",
                "content": [{"type": "text", "text": user_message}]
            },
        ],
        response_format={"type": "text"}, # 응답 타입 지정
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    return response.choices[0].message.content

### 채용공고를 입력으로 전달하고 원본 응답 확인

긴 `job_posting` 문자열은 모델이 근거로 삼을 입력 데이터이며, `output`은 함수가 반환한 JSON 문자열이다. 먼저 `type(output)`이 `str`인지와 내용이 JSON 객체처럼 보이는지 확인한 뒤 다음 셀에서 파싱한다. 실제 API 호출은 비용과 네트워크가 발생하므로 수업 환경의 키와 사용 한도를 확인한 뒤 실행한다.


In [11]:
job_posting = """
| 모집부문                      | 담당업무                                                                                                                                                                                                                                                                                               | 자격요건 및 필수사항                                                                                                                                                                                                                                                                                                                                                                 |
| ------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **AI 인프라 구축 (ML/DEVOps)** | 1. AI 모델의 배포를 위한 인프라 구축 및 운영2. 알고리즘 개발자에 대한 기술 지원3. 머신러닝 인프라의 설계 및 개발, 운영4. 모니터링, 로깅 시스템의 개발 및 구현5. 머신러닝 시스템의 학습, 배포, 테스트 환경 구축, 운영 및 자동화6. 오픈소스 기반의 플랫폼 통합 및 내재화 구축(MLflow, LakeFS, CVAT, BentoML 등)7. 보안, 인증, 권한 관리 등 운영 정책 및 관리자 기능 구축8. 시스템과 프로세스에 대한 문서 작업 |
**학력**- 대졸이상(4년제)**경력**- 신입**필수사항**- Python 프로그래밍 능력- 컨테이너 오케스트레이션 툴 활용 능력(Docker, Kubernetes 등)- Jenkins, ArgoCD, Gitlab 등 구축 및 운영 경험**우대사항**- DevOps 환경 구축 혹은 DevOps 환경에서 개발 및 배포 경험 보유자- MLflow, Kubeflow, ML 모델 서빙 플랫폼 등과 관련된 실무 경험 보유자- Kubernetes, Docker, GitOps, CI/CD, Vault, MinIO 등 클라우드 네이티브 인프라 경험 보유자 |
"""

print(job_interview(job_posting))

{
    "hard_skill": [
        {
            "question": "Python을 활용한 머신러닝 인프라 구축 경험에 대해 설명해 주세요.",
            "answer": "저는 이전 프로젝트에서 Python을 사용하여 머신러닝 모델의 배포 인프라를 구축했습니다. Flask 프레임워크를 사용해 API 서버를 개발하고, Docker를 활용하여 컨테이너화하여 이를 Kubernetes 클러스터에 배포했습니다. 이러한 과정에서 Python의 다양한 라이브러리, 특히 Pandas와 NumPy를 통해 데이터 전처리 및 모델 학습이 원활하게 진행될 수 있도록 하였습니다."
        },
        {
            "question": "컨테이너 오케스트레이션 툴인 Kubernetes를 사용해 본 경험이 있나요? 있다면 어떤 프로젝트에서 어떻게 활용했는지 말씀해 주세요.",
            "answer": "네, 저는 Kubernetes를 사용하여 대규모 머신러닝 시스템을 운영한 경험이 있습니다. 특정 프로젝트에서는 여러 모델을 동시에 서비스하기 위해 각 모델을 독립적인 Pod로 배포했습니다. Helm 차트를 사용하여 버전 관리를 철저히 하고, autoscaling 기능을 구현하여 트래픽에 따라 서버 리소스를 자동으로 조정하게 했습니다."
        },
        {
            "question": "Jenkins와 ArgoCD를 사용하여 CI/CD 파이프라인을 구축한 경험을 공유해 주세요.",
            "answer": "지난 프로젝트에서 Jenkins를 사용하여 코드 커밋 시 자동으로 빌드 및 테스트를 수행하는 CI 파이프라인을 구축했습니다. 완료된 빌드는 ArgoCD를 통해 Kubernetes 클러스터에 배포되도록 설정하여, 개발자들이 코드를 쉽게 배포할 수 있는 환경을 조성했습니다. 이를 통해 배포 시간이 50% 이상 단축될 수 있었습니다."
        }
   

### 지시문과 입력 데이터를 분리한 JSON 프롬프트

이 셀은 시스템 역할 설명과 사용자 쪽의 지시·채용공고를 분리해 같은 JSON 생성 함수를 다시 정의한다. 프롬프트 구성 요소를 분리하면 역할 규칙은 재사용하고 채용공고만 바꿀 수 있다. 두 함수 정의는 같은 이름을 사용하므로 이 셀을 실행하면 앞 셀의 정의를 덮어쓴다는 점을 확인한다.


In [12]:
def job_interview(job_posting, model='gpt-4o-mini', temperature=1, top_p=1):
    # system 메시지에는 여러 요청에서 유지할 면접관 역할만 둔다.
    system_message = """
당신은 머신러닝/딥러닝/LLM/AI서비스개발의 전문가로써, 해당분야의 으뜸가는 면접관입니다.
매번 그룹사의 인재를 발굴하기 위해 면접질문/모범답안을 작성하고 있습니다.
"""

    # 작업 지시, JSON 예시, 실제 채용공고를 한 user 메시지의 경계 표식으로 구분한다.
    # {{와 }}는 f-string이 JSON 중괄호를 변수 자리로 해석하지 않게 한다.
    user_message = f"""

<<지시사항>>
- 사용자가 제출한 채용공고의 내용을 바탕으로 예상면접질문과 답변을 작성해주세요.
- 하드스킬과 소프트스킬 두개의 섹션으로 나누어 작성해주세요.
- 각 스킬별로 질문/답변을 3개씩 만들어주세요.

<<출력형식>>
출력은 json형식으로만 반환되어야 합니다.

{{
    "hard_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ],
    "soft_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ]
}}

<< 채용공고 >>
{job_posting}
"""

    # 역할 메시지와 작업 메시지를 결합하고 JSON mode로 응답을 요청한다.
    response = client.chat.completions.create(
        model=model,  # 함수 호출 시 선택한 모델 ID이다.
        messages=[
            {
                "role": "system",
                "content": [
                    {
                    "type": "text",
                    "text": system_message
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                    "type": "text",
                    "text": user_message
                    }
                ]
            }],
            # JSON 문법은 보장하지만 hard_skill 같은 필드 존재 여부는 코드에서 별도로 검사해야 한다.
            response_format={
                "type": "json_object"
            },
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    return response.choices[0].message.content

### JSON 문자열을 Python 자료구조로 변환하기

`json.loads`는 JSON 문자열 `output`을 Python 딕셔너리로 변환한다. 변환 뒤 `hard_skill`과 `soft_skill` 키로 질문·답변 목록을 꺼내 후속 화면 표시, 파일 저장, 평가에 사용할 수 있다. JSON 문법이 깨지거나 필수 키가 없으면 오류가 나므로, API 응답 형식과 키 존재를 검증하는 것이 실무에서 중요하다.


In [14]:
import json # json -> dict 변환

output = job_interview(job_posting)
data = json.loads(output)
print(type(data)) # 타입이 dict 변환 되었는지 확인

hard_skill_qa = data["hard_skill"]
soft_skill_qa = data["soft_skill"]

print(hard_skill_qa)
print("-" * 100)
print(soft_skill_qa)

<class 'dict'>
[{'question': 'Python을 사용하여 머신러닝 모델을 배포하는 과정에 대해 설명해 주시겠습니까?', 'answer': '먼저, 머신러닝 모델을 개발하고 학습시킨 후, 이를 프로덕션 환경에 배포하기 위해 여러 단계를 거칩니다. 첫 번째로, 모델을 Python 패키지로 포장하고 REST API 형식으로 노출합니다. 그런 다음, Docker를 사용해 컨테이너화하고 Kubernetes를 통해 클러스터에서 관리합니다. 배포 후, Jenkins와 같은 CI/CD 도구를 사용하여 자동화된 배포 파이프라인을 구축하여 지속적인 배포가 가능하게 합니다.'}, {'question': '컨테이너 오케스트레이션 도구인 Kubernetes의 주요 기능과 그 사용 사례에 대해 설명해 주세요.', 'answer': 'Kubernetes는 컨테이너화된 응용 프로그램의 배포, 확장, 관리를 자동화하는 오픈소스 플랫폼입니다. 주요 기능으로는 자동화된 배포 및 롤백, 서비스 디스커버리, 로드 밸런싱, 자가 치유, 스케일링 등이 있습니다. 예를 들어, 머신러닝 모델이 들어오는 트래픽에 따라 자동으로 스케일링하도록 설정하거나 장애 발생 시 자동으로 대체 노드를 활성화하여 시스템 복구를 빠르게 할 수 있습니다.'}, {'question': 'MLflow를 이용한 모델 관리 경과를 설명해 주시겠습니까?', 'answer': 'MLflow는 머신러닝 프로젝트의 주기적인 관리 및 배포를 쉽게 만들어주는 플랫폼입니다. 모델을 학습할 때, MLflow Tracking을 사용해 실험을 기록하고, 모델 버전을 관리합니다. 모델이 배포되면, MLflow 모델 레지스트리를 통해 이전 버전과의 비교를 쉽게 할 수 있으며, UI를 통해 직접 모델을 선택하고 배포를 진행할 수 있기 때문에 데이터 과학자와 DevOps 팀 간의 협업이 향상됩니다.'}]
---------------------------------------------------------------------------